# EDA – Raw Data (before preprocessing)
Analysis of original, unprocessed reviews data.

In [ ]:
import glob
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (10, 4)
sns.set_theme(style="whitegrid")

CSV_OPTIONS = {"on_bad_lines": "skip", "encoding_errors": "replace", "engine": "python"}

review_files = sorted(glob.glob("data/raw/reviews_*.csv"))
df_raw = pd.concat(
    [pd.read_csv(f, **CSV_OPTIONS) for f in review_files], ignore_index=True
)
df_raw["LABEL-rating"] = pd.to_numeric(df_raw["LABEL-rating"], errors="coerce").astype(
    "Int32"
)

products = pd.read_csv("data/raw/product_info.csv", **CSV_OPTIONS)
df_raw = df_raw.drop(
    columns=["product_name", "brand_name", "price_usd"], errors="ignore"
)
df_raw = df_raw.merge(products, on="product_id", how="left")

print(f"Shape: {df_raw.shape}")
df_raw.head(2)

In [ ]:
TARGET = "LABEL-rating"
ID_LIKE = [
    "product_id",
    "brand_id",
    "product_name",
    "brand_name",
    "variation_desc",
    "author_id",
]

bool_cols = [
    c for c in df_raw.columns if df_raw[c].dropna().isin([0, 1]).all() and c != TARGET
]
num_cols = [
    c
    for c in df_raw.select_dtypes(include="number").columns
    if c not in bool_cols and c != TARGET
]
text_cols = [
    c
    for c in df_raw.select_dtypes(include="object").columns
    if df_raw[c].dropna().str.len().mean() > 50 and c not in ID_LIKE
]
cat_cols = [
    c
    for c in df_raw.select_dtypes(include="object").columns
    if c not in text_cols and c not in ID_LIKE
]
classes = sorted(df_raw[TARGET].dropna().unique())


print(f"Total columns:   {df_raw.shape[1]}")
print(f"  Text:          {len(text_cols)} -> {text_cols}")
print(f"  Categorical:   {len(cat_cols)} -> {cat_cols}")
print(f"  Boolean (0/1): {len(bool_cols)} -> {bool_cols}")
print(f"  Numerical:     {len(num_cols)} -> {num_cols}")
print(f"  Target:        {TARGET} (numerical, {len(classes)} classes: {classes})")

## 1. Original Rating Distribution (1–5 stars)

In [ ]:
rating_counts = df_raw["LABEL-rating"].value_counts().sort_index()
rating_pct = (rating_counts / len(df_raw) * 100).round(1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

rating_counts.plot(kind="bar", ax=axes[0], color="steelblue", edgecolor="black")
axes[0].set_title("Review count by star rating")
axes[0].set_xlabel("Stars")
axes[0].set_ylabel("Count")
axes[0].set_xticklabels([int(x) for x in rating_counts.index], rotation=0)
for bar, pct in zip(axes[0].patches, rating_pct):
    axes[0].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 3000,
        f"{pct:.1f}%",
        ha="center",
        va="bottom",
        fontsize=9,
    )

rating_pct.plot(
    kind="pie",
    ax=axes[1],
    labels=[f"{int(k)}★ ({v:.1f}%)" for k, v in zip(rating_pct.index, rating_pct)],
    autopct="",
    startangle=90,
)
axes[1].set_title("Star rating proportions")
axes[1].set_ylabel("")
plt.tight_layout()
plt.show()

print(rating_counts.to_string())

## 2. Missing Values

In [ ]:
missing = df_raw.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
missing_pct = (missing / len(df_raw) * 100).round(1)
miss_df = pd.DataFrame({"missing_count": missing, "missing_pct_%": missing_pct})
display(miss_df)

missing_pct.plot(kind="barh", figsize=(10, 7), color="steelblue")
plt.xlabel("% missing values")
plt.title("Missing values by column (raw data)")
plt.tight_layout()
plt.show()

## 3. Sparse Rows

In [ ]:
row_missing = df_raw.isnull().sum(axis=1)

dist = row_missing.value_counts().sort_index()
pct = (dist / len(df_raw) * 100).round(2)

colors = ["#e74c3c" if idx >= 32 else "steelblue" for idx in dist.index]

dist.plot(kind="bar", color=colors, edgecolor="black", figsize=(12, 4))
plt.title("Distribution of missing values per row (raw data)")
plt.xlabel("Number of missing columns in a row")
plt.ylabel("Number of rows")
plt.axvline(
    x=list(dist.index).index(32) - 0.5, color="red", linestyle="--", linewidth=1.5
)
plt.text(
    list(dist.index).index(32),
    dist.max() * 0.9,
    "← unmatched products\n   (dropped)",
    color="red",
    fontsize=9,
)
plt.tight_layout()
plt.show()

In [ ]:
sparse_rows = df_raw[df_raw.isnull().sum(axis=1) >= 32]
print(f"Sparse rows (32+ missing): {len(sparse_rows)}")
print(f"LABEL-rating NaN in these rows: {sparse_rows['LABEL-rating'].isnull().sum()}")
print(f"product_id NaN in these rows: {sparse_rows['product_id'].isnull().sum()}")

**Feature types:** 12 numerical, 15 categorical (incl. misclassified: `Unnamed: 0`, `total_feedback_count`, `submission_time` — IDs/timestamps, not real categoricals), 5 boolean flags, 3 text columns.

**Target (`LABEL-rating`):** Stored as integer (1–5 stars) → **numerical ordinal**, but treated as categorical for classification (binarised to 0/1 in preprocessing).

| Group | Columns | Missing % | Strategy |
|---|---|---|---|
| ~99% missing | `variation_desc`, `sale_price_usd`, `value_price_usd` | 97–99% | Drop columns |
| High missing (numerical) | `child_min_price`, `child_max_price`, `helpfulness` | 51–59% | Drop columns |
| Demographic (categorical) | `hair_color`, `eye_color`, `skin_tone`, `skin_type` | 10–21% | Fill with `'unknown'` |
| Text with missing | `review_title` (28%), `highlights` (11%), `ingredients` (2%) | 2–28% | Fill with `''` |
| `review_text`, `LABEL-rating` | Primary feature / target | <1% | Drop rows |
| 1127 rows (32–40 missing columns) | No match in `product_info.csv` — all product-level fields are NaN at once (merge returns NaN for entire product block) | — | Dropped via `dropna(subset=['product_id'])` in `preprocess.py` |

**Strategies for numerical missing values:** median imputation (fast, robust to outliers) or KNN imputation (more accurate, expensive). Here not needed — high-missing numerical columns are dropped as redundant with `price_usd`.


### **How missing values are handled in `scripts/preprocess.py`** (configured via `params.yaml`):

| Action | Columns | Reason |
|---|---|---|
| **Drop columns** | `variation_desc`, `sale_price_usd`, `value_price_usd`, `price_vs_value_ratio`, `child_min_price`, `child_max_price` | High missing (58–99%) |
| **Drop columns** | `helpfulness`, `total_feedback_count`, `total_neg/pos_feedback_count` | Data leakage — post-submission metrics, unknown at prediction time |
| **Drop columns** | `LABEL-rating` | Leakage — raw rating used to create the binary target |
| **Drop columns** | `author_id`, `submission_time`, `Unnamed: 0` | Identifiers / metadata, not features |
| **Drop rows** (`dropna`) | `review_text`, `product_id` | Primary feature / product missing = unusable row |
| **Drop rows** (missing target) | `LABEL-rating-category` | No label = can't train |
| **Fill → `'unknown'`** | `skin_tone`, `eye_color`, `skin_type`, `hair_color`, `tertiary_category`, `variation_type`, `variation_value`, `size` | Categorical NaN → encode as separate category |
| **Fill → `''`** | `review_title`, `highlights`, `ingredients` | Text NaN → empty string |
